vamos a ver si podemos usar el rag de azure desde este jupyter notebook que nunca he usado

In [1]:
pip install openai requests python-dotenv

  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.14.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached charset_normalizer-3.4.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manyli

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
chat_gpt_api_key = os.getenv("CHAT_GPT_4_API_KEY")
chat_gpt_api_model = os.getenv("CHAT_GPT_4_MODEL")
chat_gpt_api_endpoint = os.getenv("CHAT_GPT_4_ENDPOINT")
chat_gpt_api_temperature = float(os.getenv("CHAT_GPT_4_TEMPERATURE", 0.7))
chat_gpt_api_system_prompt = os.getenv("CHAT_GPT_4_SYSTEM_PROMPT", "You are a helpful assistant.")
chat_gpt_api_version = os.getenv("CHAT_GPT_4_API_VERSION")
index_name = os.getenv("INDEX_NAME")
index_endpoint = os.getenv("INDEX_ENDPOINT")
index_key= os.getenv("INDEX_KEY")
print(f'cargadas las variables de entorno de chat gpt 4')

In [ ]:
embedding_api_key = os.getenv("EMBEDDING_API_KEY")
embedding_api_endpoint = os.getenv("EMBEDDING_API_ENDPOINT")
embedding_api_model = os.getenv("EMBEDDING_API_MODEL")

print(f'cargadas las variables de entorno de embedding')

ahora con todo cargado tengo que hacer primero una al vector para tener el valor vectorizado  de la consulta

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=chat_gpt_api_key,
    api_version="2023-05-15",
    azure_endpoint=chat_gpt_api_endpoint
)



ahora tengo que crear  un nuevo embedding desde el cliente de azure que ha creado antes

In [ ]:
def generate_embeddings(client, text):
    response = client.embeddings.create(
        input=text,
        model = "text-embedding-ada-002"
    )
    embeddings=response.model_dump()
    print(f'Embeddings generados {embeddings}', embeddings)
    return embeddings['data'][0]['embedding']

ahora llamo al api para traer el asunto  de la vectorizacion

In [ ]:
user_query = input()
vectorised_user_query = generate_embeddings(azure_openai_client, user_query)
print(vectorised_user_query)
context=[]

#vale ya tengo el vectorizado del resultado de la consulta ahora tengo que hacer la consulta al indice y despues al llm

In [ ]:
from requests import RequestException
import requests
import json


url = f"{index_endpoint}/indexes/{index_name}/docs/search?api-version=2024-07-01"

headers = {
        "Content-Type": "application/json",
        "api-key": index_key
    }

body =   {
        "count": True,
        "select": "chunk",
        "vectorQueries": [
            {
                "vector": vectorised_user_query,
                "k": 30, # numero de ckunks maximo
                "fields": "text_vector", #filtro de datos que devuelve
                "kind": "vector"
            }
        ]
    }
print(headers)
try:
    response = requests.post(url, headers=headers, data=json.dumps(body))
    response.raise_for_status()
    print(response.status_code)
    print(response.json())
    documents = response.json()['value']
    for doc in documents:
        print(doc)
    for doc in documents:
        context.append(dict(
        {
            "chunk": doc['chunk'],
            "score": doc['@search.score']

        }
        ))
except requests.exceptions.HTTPError as err:
    print(f"Error del servidor: {err}")
except RequestException as e:
    print('request exception', e)




In [ ]:
system_prompt = f""""Eres un dungeon master que lleva toda la vida jugando a advanced dungeons and dragons e intentas que parezca misterioso el asunto"""

user_prompt = f""" the user query is: {user_query}
the context is : {context}"""

chat_completions_response = azure_openai_client.chat.completions.create(
    model = chat_gpt_api_model,
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.7
)
print(chat_completions_response)
print(chat_completions_response.choices[0].message.content)